In [ ]:
# Import required libraries

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pathlib import Path
from skimage import io
from skimage.io import imread_collection
import datetime
import re
import builtins
import os
import imageio.v2 as imageio
from IPython.display import Image, display
from collections import defaultdict

In [ ]:
# Functions to parse metadata from filenames

def get_led(filename: str) -> str:
    return filename.split("/")[-1].split("_")[0]

def get_fov(filename: str) -> int:
    return int(filename.split("/")[-1].split("_")[1][1:])

def get_str(filename: str) -> str:
    parts = filename.split("/")[-1].split("_")
    return " ".join([parts[0], parts[1], parts[-1].split(".")[0]])

def sort_by_time(file_list):
    timestamp_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}-\d{6})")
    timestamps = [timestamp_pattern.search(f.split("/")[-1]).group(1) for f in file_list]
    times = [datetime.datetime.strptime(ts, "%Y-%m-%d_%H-%M-%S-%f") for ts in timestamps]
    return [f for _, f in sorted(zip(times, file_list))]

In [ ]:
# INPUT experiment name
experiment_name = "YOUR_EXP_NAME"


# Path specification

base_path = Path("/mnt/large_storage/shared_storage/data/YOURSELF")
data_path = base_path / "data" / experiment_name

data_path = Path("/mnt/large_storage/shared_storage/data/YOUR_DATA_PATH")

output_path = base_path / "GIFs" / experiment_name
output_path.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    raise FileNotFoundError(f"data_path does not exist: {data_path}")

print(f"  data_path        = {data_path}")
print(f"  experiment_name  = {experiment_name}")
print(f"  output_path      = {output_path}")

In [ ]:

# File discovery

image_patterns = ["*.tif", "*.tiff", "*.ome.tif", "*.ome.tiff"]
filenames = []
for pattern in image_patterns:
    filenames.extend(data_path.rglob(pattern)) # recursively search for image files

filenames = sorted({str(f) for f in filenames})

print(f"Loaded {len(filenames)} images \n")

# Diagnostic: check file counts and timestamp uniqueness per LED/FOV

files_by_led = defaultdict(list)
files_by_led_by_fov = defaultdict(lambda: defaultdict(list))


for f in filenames:
    led = get_led(f)
    fov = get_fov(f)
    files_by_led[led].append(f)
    files_by_led_by_fov[led][fov].append(f)

print(f"Files per LED:")

for led in sorted(files_by_led):
    print(f"{led}: {len(files_by_led[led])} files")
    # for fov in sorted(files_by_led_by_fov[led]):
    #     print(f"  FOV {fov}: {len(files_by_led_by_fov[led][fov])} files")

In [ ]:
# # ================================================================
# # Identify corrupted TIFFs (read test)
# # ================================================================

# from tqdm import tqdm  # progress bar
# import warnings

# # Configure read test: try full read, fall back to header-only if needed
# def _read_test(path_str):
#     try:
#         # Read to force decode; if image is large, this may be slow but reliable
#         _ = io.imread(path_str)
#         return True, None
#     except Exception as exc:
#         return False, exc

# bad_files = []
# good_files = []

# # Optional: silence skimage warnings that do not indicate corruption
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     for f in tqdm(filenames, desc="Validating TIFFs"):
#         ok, exc = _read_test(f)
#         if ok:
#             good_files.append(f)
#         else:
#             bad_files.append((f, exc))

# print(f"Total files: {len(filenames)}")
# print(f"Good files : {len(good_files)}")
# print(f"Bad files  : {len(bad_files)}")

# if bad_files:
#     print("\nBad files (first 20):")
#     for f, exc in bad_files[:20]:
#         print(f"- {Path(f).name}: {type(exc).__name__} - {exc}")

# # Use this filtered list for downstream processing if desired
# filenames = good_files

In [ ]:
# INPUT LED selection (multiple may be selected, e.g., ["LED515NM", "LEDOVERHEADTIGER"])

#sel_led = ["LED515NM", "LEDOVERHEADTIGER"]
sel_led = ["LED565NM"]

# Select LED
for led in sel_led:
    if not (led.startswith('LED') and (led.endswith('NM') or led.endswith('OVERHEAD') or led.endswith('OVERHEADTIGER'))):
        raise ValueError(f"Invalid LED format: {led}. Expected format: LED<wavelength>NM or LEDOVERHEAD")

print(f"Selected LEDs = {sel_led}")

# Group filenames by position for the selected LEDs
fovs_by_led = {}
filenames_by_fov_by_led = {}

for led in sel_led:
    fovs_by_led[led] = np.unique([get_fov(f) for f in filenames if get_led(f) == led])
    filenames_by_fov_by_led[led] = {
        fov: sort_by_time([f for f in filenames if get_led(f) == led and get_fov(f) == fov])
        for fov in fovs_by_led[led]
    }

# Load images per position per LED
imgs_by_led = {}

for led in sel_led:
    imgs_by_led[led] = []
    for fov in fovs_by_led[led]:
        if filenames_by_fov_by_led[led][fov]:
            imgs_by_led[led].append(imread_collection(filenames_by_fov_by_led[led][fov]))


# Diagnostic: check file counts and timestamp uniqueness per LED/FOV

for led in sel_led:
    print(f"\n{led} - {len(fovs_by_led[led])} total files found accross {len(fovs_by_led[led])} FOVs")
    for fov in fovs_by_led[led]:
        files = filenames_by_fov_by_led[led][fov]
        if len(files) <= len(fovs_by_led[led]):
             raise ValueError(
                f"User selected {led}, but this LED has only one file per FOV. "
                f"The LED was likely only used for initialisation and did not image over time")
        print(f"  FOV {fov}: files={len(files)}")

In [ ]:
# INPUT image adjustment parameters

fov_to_display = 1
v_min_perc = [20]
v_max_perc = [99.8]

#y_min, y_max, x_min, x_max = 0, 30000, 0, 3000
y_min, y_max, x_min, x_max = 0, 3200, 0, 600

if len(v_min_perc) != len(sel_led) or len(v_max_perc) != len(sel_led):
    raise ValueError(f"Expected {len(sel_led)} values in v_min_perc and v_max_perc - one per selected LED")

print(f"Plotting FOV no {fov_to_display}")

# Display cropped images with raw and adjusted contrast per LED

for i, led in enumerate(sel_led):
    if fov_to_display not in filenames_by_fov_by_led[led]:
        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        axes[0].set_axis_off()
        axes[1].set_axis_off()
        fig.suptitle(led)
        axes[0].set_title("FOV not found")
        plt.tight_layout()
        plt.show()
        continue

    first_file = filenames_by_fov_by_led[led][fov_to_display][0]
    img = io.imread(first_file)[y_min:y_max, x_min:x_max]
    v_min = np.percentile(img, v_min_perc[i])
    v_max = np.percentile(img, v_max_perc[i])

    fig, axes = plt.subplots(1, 2, figsize=(4, 5))
    fig.suptitle(led)
    axes[0].imshow(img, cmap="viridis")
    axes[0].set_title("raw")

    axes[1].imshow(img, cmap="viridis", vmin=v_min, vmax=v_max)
    axes[1].set_title("adjusted")

    axes[0].axis("off")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Apply cropping

roi = [y_min, y_max, x_min, x_max]

# INPUT gif parameters (multiple can be selected e.g., [3, 4, 5] for FOVs 3, 4, and 5)

fovs = [3]
#fovs = np.unique(np.concatenate([fovs_by_led[led] for led in sel_led]))

fps = 4

timelim = 550 # minutes

# Generate GIFs for each FOV

timestamp_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}-\d{6})")
timestamp_format = "%Y-%m-%d_%H-%M-%S-%f"

fov_index_by_led = {
    led: {fov: i for i, fov in enumerate(fovs_by_led[led])}
    for led in sel_led
}

for p in fovs:
    frame_images_by_led = {}
    times_by_led = {}

    for led in sel_led:
        frame_files = filenames_by_fov_by_led[led][p]
        fov_index = fov_index_by_led[led][p]
        frame_images_by_led[led] = [
            image[roi[0]:roi[1], roi[2]:roi[3]] for image in imgs_by_led[led][fov_index]
        ]

        timestamps = [timestamp_pattern.search(Path(f).name).group(1) for f in frame_files]
        times_by_led[led] = [datetime.datetime.strptime(ts, timestamp_format) for ts in timestamps]

    n_frames_by_led = {led: len(frame_images_by_led[led]) for led in sel_led}
    print(f"FOV {p} frame counts: {n_frames_by_led}")

    n_frames = max(n_frames_by_led.values()) if n_frames_by_led else 0

    time_led = max(n_frames_by_led, key=n_frames_by_led.get)
    t0 = times_by_led[time_led][0]
    minutes = [(t - t0).total_seconds() / 60.0 for t in times_by_led[time_led]]
    total_minutes = minutes[-1] if minutes else 0.0

    if minutes and minutes[-1] > timelim:
        cutoff_idx = next((i for i, m in enumerate(minutes) if m > timelim), len(minutes))
        if cutoff_idx == 0:
            print(f"FOV {p} exceeds timelim before first frame; skipping.")
            continue
        n_frames = min(n_frames, cutoff_idx)
        total_minutes = timelim

    fig = plt.figure(figsize=(3 * len(sel_led), 6))
    gs = fig.add_gridspec(2, len(sel_led), height_ratios=[12, 1])
    ax_imgs = [fig.add_subplot(gs[0, i]) for i in range(len(sel_led))]
    ax_bar = fig.add_subplot(gs[1, :])

    def update(frame):
        for ax in ax_imgs:
            ax.clear()
        ax_bar.clear()

        for i, led in enumerate(sel_led):
            frame_idx = min(frame, len(frame_images_by_led[led]) - 1)
            frame_img = frame_images_by_led[led][frame_idx]
            v_min = np.percentile(frame_img, v_min_perc[i])
            v_max = np.percentile(frame_img, v_max_perc[i])
            ax_imgs[i].imshow(frame_img, cmap="viridis", vmin=v_min, vmax=v_max)
            ax_imgs[i].axis("off")
            ax_imgs[i].set_title(led)

        time_idx = min(frame, len(minutes) - 1)
        # Time bar with relative + absolute time (from LED with most frames)
        ax_bar.barh([0], [minutes[time_idx]], height=0.6, color="#ccf02e")
        ax_bar.set_xlim(0, max(total_minutes, 1e-6))
        ax_bar.set_yticks([])
        abs_time = times_by_led[time_led][time_idx].strftime("%Y-%m-%d %H:%M")
        ax_bar.set_xlabel(f"{minutes[time_idx]:.1f} min | {abs_time}")
        ax_bar.spines["top"].set_visible(False)
        ax_bar.spines["right"].set_visible(False)
        ax_bar.spines["left"].set_visible(False)

    animation = FuncAnimation(fig, update, frames=range(0, n_frames), interval=200)

    print(f"Generating FOV{p} gif...")
    gif_path = output_path / f"{experiment_name}___{sel_led}___FOV_no{p}.gif"
    animation.save(gif_path, fps=fps)
    print(f"FOV{p} gif generated and saved!")